# Machine learning 

In [ ]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier


In [ ]:
# Maak een klein datasetje met 20 rijen
rng = np.random.RandomState(42)
A = rng.uniform(50, 100, 20)
B = rng.rand(20)
C = rng.choice(['rood', 'groen', 'blauw', 'geel'], size=20)
D = rng.randint(0, 2, size=20)

# Introduceer missende waarden
missing_idx_A = rng.choice(20, size=4, replace=False)  # 4 missing in A
missing_idx_B = rng.choice(20, size=5, replace=False)  # 5 missing in B
A[missing_idx_A] = np.nan
B[missing_idx_B] = np.nan

df = pd.DataFrame({'A': A, 'B': B, 'C': C, 'D': D})

df

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df[['A', 'B', 'C']], df['D'], test_size=0.3, random_state=42)

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train[['A']])
scaler.transform(X_train[['A']])
X_test[['A']] = scaler.transform(X_test[['A']])


#scaler.fit_transform(X_train[['A']])

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
imputer.fit_transform(X_train[['A']])

imputer.transform(X_test[['A']])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[['C']])
encoder.transform(X_train[['C']])
encoder.transform(X_test[['C']])

In [ ]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

pipeline.fit_transform(X_train[['A']])
pipeline.transform(X_test[['A']])


In [ ]:
pipeline.fit_transform(X_train[['B']])

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numerieke_prepocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorische_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numerieke_prepocessor, ['A', 'B']),
    ('cat', categorische_preprocessor, ['C'])
])

preprocessor.fit_transform(X_train)
preprocessor.transform(X_test)

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

# Train
rf_pipeline.fit(X_train, y_train)

In [ ]:
#export the pipeline
import joblib
joblib.dump(rf_pipeline, 'rf_pipeline.pkl')

In [ ]:
#load the pipeline
loaded_pipeline = joblib.load('rf_pipeline.pkl')

In [ ]:
loaded_pipeline.predict(X_test)